# 特征工程

传统机器学习里，经常有一句话叫“模型上限由数据和特征决定”。这份 notebook 就是在讲，如何把原始数据变成更适合模型学习的输入表示。

它覆盖了三大类操作：
- 编码：把类别、文本变成数值向量。
- 预处理：标准化、归一化、缺失值填充。
- 选择与降维：保留更有信息量的特征。


In [1]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import numpy as np
import jieba

c:\DevEnv\python_venv\.any\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 字典特征与 One-Hot 思想

像城市这种类别特征，模型不能直接吃字符串，所以要先编码。`DictVectorizer` 的作用，就是把字典形式的类别/数值特征变成模型能处理的向量。

其中类别特征通常会被展开成 One-Hot 编码：某一维是 1，其余是 0。


In [2]:
# 字典特征提取
def dictvec():
    vector = DictVectorizer(sparse=False)

    data = vector.fit_transform([
        {'city': '北京', 'temperature': 10},
        {'city': '上海', 'temperature': 60},
        {'city': '深圳', 'temperature': 30}
    ])

    print(data)
    print(vector.get_feature_names_out())
    print(vector.inverse_transform(data))


dictvec()

[[ 0.  1.  0. 10.]
 [ 1.  0.  0. 60.]
 [ 0.  0.  1. 30.]]
['city=上海' 'city=北京' 'city=深圳' 'temperature']
[{'city=北京': np.float64(1.0), 'temperature': np.float64(10.0)}, {'city=上海': np.float64(1.0), 'temperature': np.float64(60.0)}, {'city=深圳': np.float64(1.0), 'temperature': np.float64(30.0)}]


## CountVectorizer 在做什么

词袋模型的思想是：不关心词序，先统计“哪些词出现了、出现多少次”。这样每篇文本都能变成固定长度向量。

它很简单，但已经能支持很多基础文本分类任务。


In [3]:
# 文本特征提取
def couvec():
    vector = CountVectorizer()

    res = vector.fit_transform([
        'life is short, i like python',
        'life is too long, i dislike python'
    ])

    print(vector.get_feature_names_out())
    print(res.toarray())
    print(vector.inverse_transform(res))


couvec()

['dislike' 'is' 'life' 'like' 'long' 'python' 'short' 'too']
[[0 1 1 1 0 1 1 0]
 [1 1 1 0 1 1 0 1]]
[array(['life', 'is', 'short', 'like', 'python'], dtype='<U7'), array(['life', 'is', 'python', 'too', 'long', 'dislike'], dtype='<U7')]


In [4]:
# 分词
# jieba是一个非常流行的中文分词库，可以将中文文本切分成一个个单独的词语，方便后续的文本分析和处理。
def cutword(content: str = None):
    if content:
        con = jieba.cut(content)
        con_list = list(con)
        str_con = ' '.join(con_list)
        return str_con
    else:
        con1 = jieba.cut('今天很残酷，明天更残酷，后天很美好，但绝对不能在今天晚上放弃！')
        con2 = jieba.cut('我们看到的从很远星系来的光是在几百万年之前发出的，这其中发生了什么我们不得而知，')
        con3 = jieba.cut('如果只用一种方式了解某样事物，你就不会真正了解它。')

        con_list1 = list(con1)
        con_list2 = list(con2)
        con_list3 = list(con3)

        str1 = ' '.join(con_list1)
        str2 = ' '.join(con_list2)
        str3 = ' '.join(con_list3)

        print(str1, str2, str3, sep='\n')

        return str1, str2, str3


cutword()

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\jaren\AppData\Local\Temp\jieba.cache
Loading model cost 0.333 seconds.
Prefix dict has been built successfully.


今天 很 残酷 ， 明天 更 残酷 ， 后天 很 美好 ， 但 绝对 不能 在 今天 晚上 放弃 ！
我们 看到 的 从 很 远 星系 来 的 光是在 几百万年 之前 发出 的 ， 这 其中 发生 了 什么 我们 不得而知 ，
如果 只用 一种 方式 了解 某样 事物 ， 你 就 不会 真正 了解 它 。


('今天 很 残酷 ， 明天 更 残酷 ， 后天 很 美好 ， 但 绝对 不能 在 今天 晚上 放弃 ！',
 '我们 看到 的 从 很 远 星系 来 的 光是在 几百万年 之前 发出 的 ， 这 其中 发生 了 什么 我们 不得而知 ，',
 '如果 只用 一种 方式 了解 某样 事物 ， 你 就 不会 真正 了解 它 。')

## 中文文本分词的预处理必要性

英文天然有空格，中文没有，所以在做词袋或 TF-IDF 前，必须先做分词。否则向量化器无法确定"词"的边界。


In [5]:
# 文本特征提取 汉字
# 由于汉字没有空格分隔，所以需要先进行分词处理，然后再进行特征提取。
def hanzivec():
    vector = CountVectorizer()

    words = [
        cutword('今天很残酷，明天更残酷，后天很美好，但绝对不能在今天晚上放弃！'),
        cutword('我们看到的从很远星系来的光是在几百万年之前发出的，这其中发生了什么我们不得而知，'),
        cutword('如果只用一种方式了解某样事物，你就不会真正了解它。')
    ]
    res = vector.fit_transform(words)
    
    print(vector.get_feature_names_out())
    print(res.toarray())
    print(vector.inverse_transform(res))


hanzivec()

['一种' '不会' '不得而知' '不能' '之前' '了解' '事物' '什么' '今天' '光是在' '其中' '几百万年' '发出'
 '发生' '只用' '后天' '如果' '我们' '放弃' '方式' '明天' '星系' '晚上' '某样' '残酷' '看到' '真正'
 '绝对' '美好']
[[0 0 0 1 0 0 0 0 2 0 0 0 0 0 0 1 0 0 1 0 1 0 1 0 2 0 0 1 1]
 [0 0 1 0 1 0 0 1 0 1 1 1 1 1 0 0 0 2 0 0 0 1 0 0 0 1 0 0 0]
 [1 1 0 0 0 2 1 0 0 0 0 0 0 0 1 0 1 0 0 1 0 0 0 1 0 0 1 0 0]]
[array(['今天', '残酷', '明天', '后天', '美好', '绝对', '不能', '晚上', '放弃'], dtype='<U4'), array(['我们', '看到', '星系', '光是在', '几百万年', '之前', '发出', '其中', '发生', '什么',
       '不得而知'], dtype='<U4'), array(['如果', '只用', '一种', '方式', '了解', '某样', '事物', '不会', '真正'], dtype='<U4')]


## TF-IDF 相比词频的优势

高频词不一定有区分度，比如"的""是"这种词几乎哪篇都出现。TF-IDF 会降低"到处都出现"的词的权重，提高更有区分性的词的权重。


In [6]:
# 文本特征提取 汉字 tfidf
# tfidf是对countvectorizer的升级，增加了一个逆文档频率的权重，来降低常见词的权重，提高稀有词的权重。
# tfidf 算法：tfidf = tf * idf, 其中 tf 是词频，idf 是逆文档频率，计算公式为 idf = log(总文档数 / 包含该词的文档数)
def tfidfvec():
    c1, c2, c3 = cutword()
    print(c1, c2, c3, sep='\n')

    vector = TfidfVectorizer(smooth_idf=True)
    data = vector.fit_transform([c1, c2, c3])

    print(vector.get_feature_names_out())
    print(data.toarray())
    print(vector.inverse_transform(data))


tfidfvec()

今天 很 残酷 ， 明天 更 残酷 ， 后天 很 美好 ， 但 绝对 不能 在 今天 晚上 放弃 ！
我们 看到 的 从 很 远 星系 来 的 光是在 几百万年 之前 发出 的 ， 这 其中 发生 了 什么 我们 不得而知 ，
如果 只用 一种 方式 了解 某样 事物 ， 你 就 不会 真正 了解 它 。
今天 很 残酷 ， 明天 更 残酷 ， 后天 很 美好 ， 但 绝对 不能 在 今天 晚上 放弃 ！
我们 看到 的 从 很 远 星系 来 的 光是在 几百万年 之前 发出 的 ， 这 其中 发生 了 什么 我们 不得而知 ，
如果 只用 一种 方式 了解 某样 事物 ， 你 就 不会 真正 了解 它 。
['一种' '不会' '不得而知' '不能' '之前' '了解' '事物' '什么' '今天' '光是在' '其中' '几百万年' '发出'
 '发生' '只用' '后天' '如果' '我们' '放弃' '方式' '明天' '星系' '晚上' '某样' '残酷' '看到' '真正'
 '绝对' '美好']
[[0.         0.         0.         0.25819889 0.         0.
  0.         0.         0.51639778 0.         0.         0.
  0.         0.         0.         0.25819889 0.         0.
  0.25819889 0.         0.25819889 0.         0.25819889 0.
  0.51639778 0.         0.         0.25819889 0.25819889]
 [0.         0.         0.26726124 0.         0.26726124 0.
  0.         0.26726124 0.         0.26726124 0.26726124 0.26726124
  0.26726124 0.26726124 0.         0.         0.         0.53452248
  0.         0.         0.         0.2672612

## 标准化与归一化的区分

- 归一化：通常压到固定区间，比如 `0~1`。
- 标准化：通常变成均值 0、方差 1。

哪种更合适，取决于模型和数据分布。基于距离的算法尤其依赖尺度统一。


In [7]:
# 特征缩放
# 归一化：将数据缩放到一个指定的范围内，通常是0到1之间。常用的归一化方法是Min-Max Scaling。
# 公式为：X_scaled = (X - X_min) / (X_max - X_min)，其中X是原始数据，X_min和X_max分别是数据的最小值和最大值。
# 标准化：将数据缩放到均值为0，标准差为1的分布。常用的标准化方法是Z-score Standardization。
# 公式为：X_scaled = (X - μ) / σ，其中X是原始数据，μ是数据的均值，σ是数据的标准差。
def mm():
    mm = MinMaxScaler(feature_range=(0, 1))

    data = mm.fit_transform([
        [90, 2, 10, 40],
        [60, 4, 15, 30],
        [75, 3, 13, 35]
    ])

    print(data)


mm()

print('-' * 50)


# 方差公式：σ² = Σ(xi - μ)² / n
# 标准化后的数据的方差为1，均值为0，标准差为1。
def stand():
    std = StandardScaler()

    data = std.fit_transform([
        [90, 2, 10, 40],
        [60, 4, 15, 30],
        [75, 3, 13, 35]
    ])

    print('方差:', std.var_)
    print('均值:', std.mean_)
    print('标准差:', std.scale_)
    print('标准化后的数据:\n', data)

    print('标准化后的数据的方差:', np.var(data, axis=0))
    print('标准化后的数据的均值:', np.mean(data, axis=0))


stand()

[[1.  0.  0.  1. ]
 [0.  1.  1.  0. ]
 [0.5 0.5 0.6 0.5]]
--------------------------------------------------
方差: [150.           0.66666667   4.22222222  16.66666667]
均值: [75.          3.         12.66666667 35.        ]
标准差: [12.24744871  0.81649658  2.05480467  4.0824829 ]
标准化后的数据:
 [[ 1.22474487 -1.22474487 -1.29777137  1.22474487]
 [-1.22474487  1.22474487  1.13554995 -1.22474487]
 [ 0.          0.          0.16222142  0.        ]]
标准化后的数据的方差: [1. 1. 1. 1.]
标准化后的数据的均值: [0.00000000e+00 0.00000000e+00 2.77555756e-16 0.00000000e+00]


## 缺失值处理的策略选择

缺失值的填充方式，实际上隐含了对数据分布的假设：
- 均值：适合大致对称分布。
- 中位数：对异常值更稳。
- 众数：适合类别特征。


In [8]:
# 缺失值处理
def im():
    # missing_values：指定缺失值的标记，默认为np.nan。
    # strategy：指定填充策略，常用的有'mean'（均值填充）、'median'（中位数填充）和'most_frequent'（众数填充）。
    im = SimpleImputer(missing_values=np.nan, strategy='mean')

    data = im.fit_transform([
        [90, 2, 10, 40],
        [60, np.nan, 15, 30],
        [75, 3, np.nan, 35]
    ])

    print(data)


im()

[[90.   2.  10.  40. ]
 [60.   2.5 15.  30. ]
 [75.   3.  12.5 35. ]]


## 方差筛选与 PCA 的对比

- 方差筛选：删除变化几乎没有的特征。
- PCA：把多个原始特征重新组合成少量主成分。

前者更偏向过滤，后者更偏向压缩和重表示。


In [9]:
# 特征选择
def var():
    # threshold：指定方差的阈值，默认为0.0。只有当特征的方差大于该阈值时，才会被保留。
    var = VarianceThreshold(threshold=0.1)

    data = var.fit_transform([
        [0, 2, 0, 3],
        [0, 1, 4, 3],
        [0, 1, 1, 3]
    ])

    print(data)
    print('特征选择结果:', var.get_support(True))

 
var()

[[2 0]
 [1 4]
 [1 1]]
特征选择结果: [1 2]


In [10]:
# 特征降维 (PCA 主成分分析)
def pca():
    original_date = np.array([
        [2, 8, 4, 5],
        [6, 3, 0, 8],
        [5, 4, 9, 1]
    ])
    print('原始数据:\n', original_date)
    print('原始数据方差和:', np.var(original_date, axis=0).sum())

    # n_components：指定要保留的主成分数量，可以是整数（表示保留的主成分数量）或浮点数（表示保留的方差比例）。
    # 当n_components为0.9时，表示保留90%的方差。
    pca = PCA(n_components=0.9)

    data = pca.fit_transform(original_date)
    print('降维后的数据:\n', data)
    print('降维后的数据方差和:', np.var(data, axis=0).sum())
    print('各主成分解释的方差比例:', pca.explained_variance_ratio_)


pca()

原始数据:
 [[2 8 4 5]
 [6 3 0 8]
 [5 4 9 1]]
原始数据方差和: 29.333333333333336
降维后的数据:
 [[-1.28620952e-15  3.82970843e+00]
 [-5.74456265e+00 -1.91485422e+00]
 [ 5.74456265e+00 -1.91485422e+00]]
降维后的数据方差和: 29.333333333333332
各主成分解释的方差比例: [0.75 0.25]
